In [0]:
from pyspark.sql.functions import *

In [0]:
dim_customer = spark.table('lakehouse.silver.s_cust_info')
dim_product = spark.table('lakehouse.silver.s_prd_info_1')
fact_sales = spark.table('lakehouse.silver.s_sales_details')

In [0]:
display(dim_customer.head(2))

cust_id,cust_key,first_name,last_name,marital_status,gender,create_date
11000,AW00011000,Jon,Yang,Single,Female,2025-10-06
11001,AW00011001,Eugene,Huang,Single,Female,2025-10-06


In [0]:
display(dim_product.limit(2))

prd_id,prd_key,prd_name,prd_cost,prd_line
408,HB-R956,HL Road Handlebars,53,R
486,ST-1401,All-Purpose Bike Stand,59,M


In [0]:
display(fact_sales.limit(2))

sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price
SO43697,BK-R93R-62,21768,2010-12-29,2011-01-05,2011-01-10,3578,1,3578
SO43698,BK-M82S-44,28389,2010-12-29,2011-01-05,2011-01-10,3400,1,3400


# Creating Star Schema

### dim_customer

In [0]:
dim_customer = dim_customer.select(
    "cust_id",
    "cust_key",
    "first_name",
    "last_name",
    "gender",
    "marital_status"
).dropDuplicates(["cust_id"])

In [0]:
dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lakehouse.gold.dim_customer")

### dim_product

In [0]:
dim_product = dim_product.select(
    "prd_id",
    "prd_key",
    "prd_name",
    "prd_cost",
    "prd_line"
).dropDuplicates(["prd_id"])

In [0]:
dim_product.show()

+------+----------+--------------------+--------+--------+
|prd_id|   prd_key|            prd_name|prd_cost|prd_line|
+------+----------+--------------------+--------+--------+
|   408|   HB-R956|  HL Road Handlebars|      53|       R|
|   486|   ST-1401|All-Purpose Bike ...|      59|       M|
|   504|FR-T67U-58|LL Touring Frame ...|     200|       T|
|   503|FR-T67U-54|LL Touring Frame ...|     200|       T|
|   502|FR-T67U-50|LL Touring Frame ...|     200|       T|
|   501|   RD-2308|     Rear Derailleur|      54|       R|
|   500|FR-T98U-60|HL Touring Frame ...|     602|       T|
|   499|FR-T98U-54|HL Touring Frame ...|     602|       T|
|   498|FR-T98U-50|HL Touring Frame ...|     602|       T|
|   497|FR-T98U-46|HL Touring Frame ...|     602|       T|
|   496|FR-T98Y-54|HL Touring Frame ...|     602|       T|
|   495|FR-T98Y-50|HL Touring Frame ...|     602|       T|
|   494|FR-T98Y-46|HL Touring Frame ...|     602|       T|
|   493|FR-T67Y-62|LL Touring Frame ...|     200|       

In [0]:
dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lakehouse.gold.dim_product")

### fact_sales

In [0]:
fact_sales = fact_sales.select(
    "sls_ord_num",
    "sls_prd_key",
    "sls_cust_id",
    "sls_order_dt",
    "sls_ship_dt",
    "sls_due_dt",
    "sls_sales",
    "sls_quantity",
    "sls_price"
)

In [0]:
fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lakehouse.gold.fact_sales")


### checking if joins are working correctly

In [0]:
fact_sales.join(
    dim_customer,
    fact_sales.sls_cust_id == dim_customer.cust_id,
    "left_anti"
).count()

0

In [0]:
fact_sales.join(
    dim_product,
    fact_sales.sls_prd_key == dim_product.prd_key,
    "left_anti"
).count()

0

In [0]:
fact_sales.join(
    dim_product,
    fact_sales.sls_prd_key == dim_product.prd_key,
    "left_anti"
).count()

0

In [0]:
dim_product.display()

In [0]:
from pyspark.sql.functions import *

In [0]:
dim_product.join(fact_sales, fact_sales.sls_prd_key == dim_product.prd_key, 'inner')\
    .groupBy('prd_name')\
    .agg(avg('sls_price')).show()

+--------------------+------------------+
|            prd_name|    avg(sls_price)|
+--------------------+------------------+
|    Road-150 Red- 44|            3578.0|
|  Road-250 Black- 44| 2318.760147601476|
|Sport-100 Helmet-...| 34.99520383693046|
|Sport-100 Helmet-...| 35.04941176470588|
|Half-Finger Glove...|24.013544018058692|
|Touring-3000 Blue...|             742.0|
|Touring-1000 Blue...|            2384.0|
|Road-350-W Yellow...|            1701.0|
|Touring-3000 Blue...|             742.0|
|Touring-3000 Yell...|             742.0|
|Mountain-100 Silv...|            3400.0|
|Road-550-W Yellow...|            1088.0|
|  Road-750 Black- 52|             540.0|
|Fender Set - Moun...|22.003771805752002|
|Half-Finger Glove...|24.018036072144287|
| Patch Kit/8 Patches|2.0206831714196176|
|Touring-1000 Blue...|            2384.0|
|Mountain-400-W Si...|             769.0|
|Mountain-400-W Si...|             769.0|
|Mountain-500 Blac...|             540.0|
+--------------------+------------

In [0]:
fact_sales.createOrReplaceTempView("fact_sales")
dim_product.createOrReplaceTempView("dim_product")
dim_customer.createOrReplaceTempView("dim_customer")

In [0]:
product_sales_summary = spark.sql("""
          
                                    select 
                                        prd_name,
                                        sum(sls_price) as total_price
                                    from fact_sales as f
                                    inner join dim_product as d
                                    on f.sls_prd_key = d.prd_key
                                    group by prd_name
                                                                        """)
product_sales_summary.display()

prd_name,total_price
Road-150 Red- 44,1005418
Road-250 Black- 44,628384
Sport-100 Helmet- Black,72965
Sport-100 Helmet- Blue,74480
Half-Finger Gloves- L,10638
Touring-3000 Blue- 54,40810
Touring-1000 Blue- 60,350448
Road-350-W Yellow- 44,367416
Touring-3000 Blue- 50,35616
Touring-3000 Yellow- 44,43778
